# 3. Tensor Parallelism

Tensor parallelism splits a **single large layer** across multiple GPUs.

**Core idea**
- each GPU stores only part of one weight matrix
- every GPU computes part of the same layer
- partial results are combined into the final answer

This is more advanced than data or pipeline parallelism because a
single layer is now distributed.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Two common ways to split a linear layer

1. **Column parallelism**
   Each GPU produces a slice of the output features.
2. **Row parallelism**
   Each GPU handles a slice of the input features, and the partial
   outputs are added together.

We will show both with one small `nn.Linear` layer.

In [ ]:
in_features = 6
out_features = 8
batch_size = 3
num_shards = 2

linear = nn.Linear(in_features, out_features)
x = torch.randn(batch_size, in_features)
full_output = linear(x)

print("Input shape:", tuple(x.shape))
print("Weight shape:", tuple(linear.weight.shape))
print("Full output shape:", tuple(full_output.shape))

In [ ]:
# Column parallelism: split output features across shards.
column_weight_shards = linear.weight.chunk(num_shards, dim=0)
column_bias_shards = linear.bias.chunk(num_shards, dim=0)

column_outputs = []
for shard_id, (weight_shard, bias_shard) in enumerate(
    zip(column_weight_shards, column_bias_shards)
):
    local_output = F.linear(x, weight_shard, bias_shard)
    column_outputs.append(local_output)
    print(
        f"Column shard {shard_id}: "
        f"weight={tuple(weight_shard.shape)}, output={tuple(local_output.shape)}"
    )

column_parallel_output = torch.cat(column_outputs, dim=-1)

print()
print(
    "Max difference after column parallelism:",
    (full_output - column_parallel_output).abs().max().item(),
)
assert torch.allclose(full_output, column_parallel_output, atol=1e-7)

In [ ]:
# Row parallelism: split input features across shards.
x_shards = x.chunk(num_shards, dim=-1)
row_weight_shards = linear.weight.chunk(num_shards, dim=1)

partial_outputs = []
for shard_id, (x_shard, weight_shard) in enumerate(zip(x_shards, row_weight_shards)):
    local_partial = F.linear(x_shard, weight_shard, bias=None)
    partial_outputs.append(local_partial)
    print(
        f"Row shard {shard_id}: "
        f"input={tuple(x_shard.shape)}, weight={tuple(weight_shard.shape)}"
    )

row_parallel_output = sum(partial_outputs) + linear.bias

print()
print(
    "Max difference after row parallelism:",
    (full_output - row_parallel_output).abs().max().item(),
)
assert torch.allclose(full_output, row_parallel_output, atol=1e-7)
print("Both tensor-parallel splits reproduce the full layer output.")

## Takeaways

- Tensor parallelism is useful when a single layer is too large for
  one GPU.
- Column parallelism combines outputs with **concatenation**.
- Row parallelism combines outputs with **summation**.
- This usually needs more communication than pure data parallelism.